In [1]:
from jbfuncs.gbmaker import sample_gb_energy, SpheregbBOMaker

In [2]:
from pymatgen.core.structure import Structure
LLZO = Structure.from_file('sorted_optimized_bulk_POSCAR')

In [3]:
import numpy as np
sample_job = SpheregbBOMaker(
                    name = 'mzy_bo_test', # job name
                    trials = 10, # number of optimization trials
                    crystal_structure = LLZO, # structure
                    check_point_file = '/home/xys/check_points/dpa/LLZO-c.pb', #check point file
                    bulk_energy_traj_file = '/home/xys/check_points/dpa/bulk_energy.lammpstrj', #bulk traj
                    sphere_R = 50, # sphere radius
                    vaccum_thickness = 20, # vaccum thickness
                    gb_r = 40, # gb radius
                    rot_axis = [0,-2,1],
                    rot_angle = 180/180 * np.pi,
                    normal = [0,-2,1])
test_job = sample_job.make()
test_job.update_metadata({'name':'mzy_bo_test_0'}) # you need set this to read result

In [4]:
# which gpu to use
from qtoolkit.core.data_objects import QResources
resources = QResources(
        nodes = 1,
        processes_per_node = 1,
        gpus_per_job = 1,
        scheduler_kwargs = {
            "partition": 'gpu2',
            "qverbatim": "#SBATCH --cpus-per-gpu=10"})

In [5]:
from jobflow_remote import submit_flow
from jobflow import Flow

In [6]:
#submit
submit_flow(Flow([test_job]), worker ='llzo_worker',
            resources = resources, project = 'std')

['276']

In [1]:
# read result
from jobflow_remote.config.jobconfig import load_job_store

job_store = load_job_store('std')
with job_store as js:
    result = js.query_one({'metadata.name': 'mzy_bo_test_0'})['output'] # the metadata you defined before

In [8]:
# best params
best_x = result['x'][result['y'].index(min(result['y']))]
best_x

[0.0,
 3.3106886802381457,
 0.6938671566171617,
 0.0,
 8.00857867476411,
 2.448364608190626,
 2.8771182846289425]

In [7]:
x1, y1, z1, x2, y2, z2, gap = best_x

In [9]:
xyz_1 = [x1, y1, z1]
xyz_2 = [x2, y2, z2]